# Loopback Capture Example

This notebook shows how to use `Measurement.capture_loopback(...)` to capture full-span loopback waveforms from READ_IN and MNTR_IN channels.

`capture_loopback` temporarily switches RF ports to loopback mode, executes the schedule, and restores the original RF-switch states automatically.

In [1]:
%load_ext autoreload
%autoreload 2

import qubex as qx
import numpy as np
from qxpulse import Blank, FlatTop, Gaussian, PulseSchedule

from qubex.measurement import Measurement

In [2]:
# Initialize measurement session (replace paths/chip/qubits for your environment).
session = qx.Experiment(
    chip_id="64Qv3",
    qubits=["Q62", "Q63"],
    config_dir="/home/shared/qubex-config/64Qv3/config",
    params_dir="/home/shared/qubex-config/64Qv3/params",
)

date: 2026-02-26 16:08:27
python: 3.10.18
qubex: 1.5.0a2+gf922faf4
env: /home/machino/qubex/.venv
config: /home/shared/qubex-config/64Qv3/config
params: /home/shared/qubex-config/64Qv3/params
chip: 64Qv3 (2023-1st-64Q-No14-run3 chip (1,0))
qubits: ['Q62', 'Q63']
muxes: ['MUX15']
boxes: ['U15A']


In [3]:
# Hardware connection is required for actual capture.
session.connect()

Successfully connected.


In [ ]:
session.tool.get_quel1_box("U15A")

In [ ]:
q0 = session.qubits[0]
read0 = session.experiment_system.resolve_read_label(q0)

schedule = PulseSchedule()
with schedule as s:
    # Drive pulse region
    s.add(q0, Gaussian(duration=32, amplitude=0.03, sigma=8))
    s.add(q0, Blank(duration=128))

    # Readout pulse region
    s.barrier()
    s.add(read0, FlatTop(duration=512, amplitude=0.1, tau=32))

schedule.plot()

In [ ]:
session.box_ids

In [ ]:
# Full-span loopback capture over the whole schedule duration.
result = session.capture_loopback(
    schedule=schedule,
    n_shots=1024,
)
result

In [ ]:
print(f"sampling_period_ns = {result.sampling_period_ns}")
for target, captures in result.data.items():
    shapes = [np.asarray(cap).shape for cap in captures]
    print(target, shapes)

In [ ]:
# Visualize each captured waveform.
result.plot()